Environment & LangSmith Config

In [1]:
import os
import re
import psycopg2
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Dict, Any, Optional
import operator

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_cohere import CohereEmbeddings
import cohere

load_dotenv()

# LangSmith Setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "OmniQuery_Master_Supervisor"

# Clients Setup
cohere_api_key = os.getenv("COHERE_API_KEY")
embeddings_model = CohereEmbeddings(model="embed-english-v3.0", cohere_api_key=cohere_api_key)
cohere_client = cohere.Client(api_key=cohere_api_key)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME", "enterprise_hub"),
    "user": os.getenv("DB_USER", "admin"),
    "password": os.getenv("DB_PASSWORD", "secretpassword"),
}

RELEVANCE_THRESHOLD = 0.25
MAX_RETRIES = 3

d:\rag-projects\OmniQuery--multiagent-sql-rag-analyst\omniquery-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AgentState

In [2]:
class AgentState(TypedDict):
    # Core User Intent
    messages: Annotated[List[Dict[str, Any]], operator.add]
    route: str  # "sql", "rag", or "both"
    
    # SQL Subgraph State
    sql_query: Optional[str]
    sql_result: Optional[List[Dict[str, Any]]]
    sql_error: Optional[str]
    sql_retry_count: int
    sql_validation_error: Optional[str]
    sql_status: Optional[str]  # "success", "blocked", "failed"

    # RAG Subgraph State
    rag_context: Optional[List[Dict[str, Any]]]
    rag_error: Optional[str]

    # Synthesized Output
    final_response: Optional[str]

SQL Agent Node

In [3]:
FORBIDDEN_KEYWORDS = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", "CREATE", "GRANT", "REVOKE"]

def get_database_schema() -> str:
    return """
    Table: product_sales
    Columns:
      - id (SERIAL PRIMARY KEY)
      - region (VARCHAR(50)) -- 'North America', 'Europe', 'Asia-Pacific'
      - product_line (VARCHAR(100)) -- 'AR Interior Designer Pro (License)', 'Computer Vision API Tracker'
      - revenue (NUMERIC(12, 2))
      - units_sold (INT)
      - fiscal_quarter (VARCHAR(10)) -- 'Q1-2026'
    """

def sql_agent_node(state: AgentState) -> Dict[str, Any]:
    """Encapsulates the complete SQL generation, validation, execution, and retry loop."""
    user_query = state["messages"][-1]["content"]
    retry_count = 0
    last_error = None
    sql_query = ""

    while retry_count <= MAX_RETRIES:
        # 1. Generate SQL
        system_prompt = f"""You are a PostgreSQL expert for OmniQuery.
Database Schema:
{get_database_schema()}

Rules:
1. Generate read-only PostgreSQL SELECT queries ONLY.
2. Use ONLY the table and columns listed above.
3. Return ONLY raw SQL, no markdown formatting.
4. If asked about non-existent entities, return: SELECT 0 WHERE 1=0.
"""
        if last_error:
            prompt = system_prompt + f"\nPrevious SQL failed: {sql_query}\nError: {last_error}\nFix the query for: {user_query}"
        else:
            prompt = system_prompt + f"\nUser Request: {user_query}"

        response = llm.invoke(prompt)
        sql_query = response.content.strip().replace("```sql", "").replace("```", "").strip()

        # 2. Validate SQL
        sql_upper = sql_query.upper()
        blocked = any(re.search(rf"\b{kw}\b", sql_upper) for kw in FORBIDDEN_KEYWORDS)
        if blocked or sql_upper.count(";") > 1:
            return {
                "sql_query": sql_query,
                "sql_result": None,
                "sql_status": "blocked",
                "sql_error": "Forbidden operations or multi-statement query blocked."
            }

        # 3. Execute SQL
        try:
            conn = psycopg2.connect(**DB_CONFIG)
            conn.set_session(readonly=True)
            cursor = conn.cursor(cursor_factory=RealDictCursor)
            cursor.execute(sql_query)
            results = cursor.fetchall()
            conn.close()

            return {
                "sql_query": sql_query,
                "sql_result": [dict(row) for row in results],
                "sql_status": "success",
                "sql_error": None,
                "sql_retry_count": retry_count
            }
        except Exception as e:
            last_error = str(e).strip()
            retry_count += 1

    return {
        "sql_query": sql_query,
        "sql_result": None,
        "sql_status": "failed",
        "sql_error": last_error,
        "sql_retry_count": retry_count
    }

RAG Agent Node

In [4]:
def rag_agent_node(state: AgentState) -> Dict[str, Any]:
    """Encapsulates Hybrid Search + RRF + Cohere Reranking."""
    user_query = state["messages"][-1]["content"]

    try:
        # 1. Embed Query
        query_vector = embeddings_model.embed_query(user_query)
        vector_str = "[" + ",".join(map(str, query_vector)) + "]"

        conn = psycopg2.connect(**DB_CONFIG)
        conn.set_session(readonly=True)
        cursor = conn.cursor(cursor_factory=RealDictCursor)

        # Dense Search
        dense_sql = "SELECT id, document_name, chunk_content, 1 - (embedding <=> %s::vector) AS score FROM document_chunks ORDER BY embedding <=> %s::vector LIMIT 10;"
        cursor.execute(dense_sql, (vector_str, vector_str))
        dense_rows = cursor.fetchall()

        # Sparse OR Search
        sparse_sql = """
            SELECT id, document_name, chunk_content, ts_rank_cd(fts_tokens, query) AS score
            FROM document_chunks, to_tsquery('english', REPLACE(plainto_tsquery('english', %s)::text, '&', '|')) query
            WHERE fts_tokens @@ query ORDER BY score DESC LIMIT 10;
        """
        cursor.execute(sparse_sql, (user_query,))
        sparse_rows = cursor.fetchall()
        conn.close()

        # RRF Fusion (k=60)
        rrf_map = {}
        for i, r in enumerate(dense_rows):
            doc_id = str(r["id"])
            rrf_map[doc_id] = {"doc": r["document_name"], "text": r["chunk_content"], "score": 1.0 / (60 + i + 1)}
        for i, r in enumerate(sparse_rows):
            doc_id = str(r["id"])
            if doc_id in rrf_map:
                rrf_map[doc_id]["score"] += 1.0 / (60 + i + 1)
            else:
                rrf_map[doc_id] = {"doc": r["document_name"], "text": r["chunk_content"], "score": 1.0 / (60 + i + 1)}

        candidates = sorted(rrf_map.values(), key=lambda x: x["score"], reverse=True)[:10]

        if not candidates:
            return {"rag_context": [], "rag_error": None}

        # 2. Cohere Rerank + Thresholding
        docs_text = [c["text"] for c in candidates]
        rerank_res = cohere_client.rerank(model="rerank-english-v3.0", query=user_query, documents=docs_text, top_n=3)

        valid_contexts = []
        for item in rerank_res.results:
            if item.relevance_score >= RELEVANCE_THRESHOLD:
                orig = candidates[item.index]
                valid_contexts.append({
                    "document_name": orig["doc"],
                    "content": orig["text"],
                    "rerank_score": round(item.relevance_score, 4)
                })

        return {"rag_context": valid_contexts, "rag_error": None}

    except Exception as e:
        return {"rag_context": [], "rag_error": str(e)}

Supervisor Router Node

In [5]:
def supervisor_node(state: AgentState) -> Dict[str, Any]:
    """Determines whether the user request requires SQL, RAG, or BOTH."""
    user_query = state["messages"][-1]["content"]

    prompt = f"""You are the Master Supervisor Router for OmniQuery, an enterprise analytics system.
Classify the following query into exactly one category:

1. 'sql': The query asks strictly for numbers, revenue metrics, sales unit counts, or database aggregations.
2. 'rag': The query asks for explanations, strategy, roadmap details, qualitative causes, or PDF report context.
3. 'both': The query requires BOTH numerical database figures AND explanatory context from reports (e.g., "What was European revenue and why did it decline?").

Output ONLY the category name ('sql', 'rag', or 'both') in lowercase. No markdown, no punctuation.

User Query: {user_query}
"""
    response = llm.invoke(prompt)
    route = response.content.strip().lower().replace("'", "").replace('"', "")

    if route not in ["sql", "rag", "both"]:
        route = "both"  # Fallback to safety

    return {"route": route}


def route_supervisor(state: AgentState) -> List[str]:
    """Returns target node names. Returning a list of 2 items triggers parallel execution in LangGraph."""
    route = state.get("route", "both")
    if route == "sql":
        return ["sql_agent"]
    elif route == "rag":
        return ["rag_agent"]
    else:
        # Fan-Out Parallel Execution!
        return ["sql_agent", "rag_agent"]

Master Synthesizer Node

In [6]:
def synthesizer_node(state: AgentState) -> Dict[str, Any]:
    """Fuses SQL results and RAG contexts into a seamless executive briefing."""
    user_query = state["messages"][-1]["content"]
    route = state.get("route", "both")
    sql_res = state.get("sql_result")
    sql_err = state.get("sql_error")
    rag_ctx = state.get("rag_context")

    prompt = f"""You are the Chief Analytics Officer presenting an executive briefing for OmniQuery.
Answer the user's question by combining the available evidence.

User Question: {user_query}
Routing Mode: {route.upper()}

=== Structured Data (SQL Database) ===
{sql_res if sql_res else 'No structured data requested or found.'}
{f'SQL Note/Error: {sql_err}' if sql_err else ''}

=== Unstructured Evidence (Document Excerpts) ===
{rag_ctx if rag_ctx else 'No document context requested or found.'}

Synthesis Guidelines:
1. Provide a direct, professional, and natural response.
2. If SQL data exists, state numerical findings and format currency clearly.
3. If document excerpts exist, reference source PDF names naturally (e.g., "Per Q1_2026_Executive_Summary.pdf...").
4. If neither source contains data, state politely that records do not exist.
5. Do NOT mention internal system terms like "SQL query", "RAG agent", "JSON", or "subgraph".
"""
    response = llm.invoke(prompt)
    return {"final_response": response.content.strip()}

Assemble Master Graph

In [7]:
master_builder = StateGraph(AgentState)

# Add Nodes
master_builder.add_node("supervisor", supervisor_node)
master_builder.add_node("sql_agent", sql_agent_node)
master_builder.add_node("rag_agent", rag_agent_node)
master_builder.add_node("synthesizer", synthesizer_node)

# Set Entry Point
master_builder.set_entry_point("supervisor")

# Conditional Router (Triggers Parallel Branches when returning a list)
master_builder.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "sql_agent": "sql_agent",
        "rag_agent": "rag_agent"
    }
)

# Fan-In Join: Both agents point directly to Synthesizer
master_builder.add_edge("sql_agent", "synthesizer")
master_builder.add_edge("rag_agent", "synthesizer")

# Exit Edge
master_builder.add_edge("synthesizer", END)

# Compile Master Graph
master_graph = master_builder.compile()

Test

In [9]:
def run_master_test_matrix():
    tests = [
        {"name": "Test 1 — Pure SQL Route", "query": "What is the total revenue in North America across all products?"},
        {"name": "Test 2 — Pure RAG Route", "query": "Why did European AR Interior Designer Pro sales experience a slowdown?"},
        {"name": "Test 3 — Both / Hybrid Parallel Route", "query": "What was the revenue for AR Interior Designer Pro in Europe and what caused the performance in that region?"},
        {"name": "Test 4 — Out of Bounds / Unknown Route", "query": "What was Apple's iPhone sales revenue in 2025?"}
    ]

    for test in tests:
        print(f"\n{'='*75}")
        print(f"▶️ {test['name']}")
        print(f"User Query: {test['query']}")
        print(f"{'-'*75}")

        initial_state = {
            "messages": [{"role": "user", "content": test["query"]}],
            "sql_retry_count": 0
        }

        output = master_graph.invoke(initial_state)

        print(f"🔀 Routed Path:     {output.get('route').upper()}")
        print(f"📊 SQL Rows Fetched: {len(output.get('sql_result') or []) if output.get('sql_result') else 'None'}")
        print(f"📄 RAG Docs Passed:  {len(output.get('rag_context') or [])}")
        print(f"{'-'*75}")
        print(f"💬 Final Executive Answer:\n{output.get('final_response')}")

if __name__ == "__main__":
    run_master_test_matrix()


▶️ Test 1 — Pure SQL Route
User Query: What is the total revenue in North America across all products?
---------------------------------------------------------------------------
🔀 Routed Path:     SQL
📊 SQL Rows Fetched: 1
📄 RAG Docs Passed:  0
---------------------------------------------------------------------------
💬 Final Executive Answer:
Our analysis indicates that the total revenue in North America across all products is $2,100,000.00.

▶️ Test 2 — Pure RAG Route
User Query: Why did European AR Interior Designer Pro sales experience a slowdown?
---------------------------------------------------------------------------
🔀 Routed Path:     RAG
📊 SQL Rows Fetched: None
📄 RAG Docs Passed:  1
---------------------------------------------------------------------------
💬 Final Executive Answer:
The slowdown in European AR Interior Designer Pro sales can be attributed to regional regulatory delays in spatial computing compliance, as noted in the Q1_2026_Executive_Summary.pdf. Specifi